# Free-field reconstruction

Three-channel pressure–velocity reconstruction at regular 6× spatial downsampling, with 100 reverse steps and seed 123. The reconstruction example and multi-trial comparison use the physics weights listed in [case settings](../../docs/cases.md).

Run cells in order. Sampling counts are reverse-diffusion steps; checkpoint numbers identify training state. Data and checkpoints are installed under the repository root.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'free_field'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
import importlib
import sampler.utils
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = '2DSound'
version = 'v22'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint_200.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
class DataScalar:

    def __init__(self, mean, std, device='cpu'):
        self.mean_np = mean
        self.std_np = std
        self.device = device
        self.mean_torch = torch.tensor(mean).float().to(device)
        self.std_torch = torch.tensor(std).float().to(device)
        if self.mean_torch.ndim == 5:
            self.mean_torch = self.mean_torch.permute(0, 1, 4, 2, 3)
            self.std_torch = self.std_torch.permute(0, 1, 4, 2, 3)

    def normalize(self, x_np):
        mean_t = self.mean_np.transpose(0, 1, 4, 2, 3)
        std_t = self.std_np.transpose(0, 1, 4, 2, 3)
        return (x_np - mean_t) / std_t

    def unnormalize(self, x):
        if x.device != self.mean_torch.device:
            self.mean_torch = self.mean_torch.to(x.device)
            self.std_torch = self.std_torch.to(x.device)
        return x * self.std_torch + self.mean_torch

def select_test_sample_by_freq(data_raw, frequency_hz=375):
    frequencies = [125, 135, 175, 225, 275, 325, 375, 415, 425]
    if frequency_hz not in frequencies:
        raise ValueError(f"Choose a frequency from {frequencies}")
    index = frequencies.index(frequency_hz)
    print(f"Test frequency: {frequency_hz} Hz")
    return data_raw[index:index + 1]


In [ ]:
import sys
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from models.fno import FNO2d
from models.lno import LNO2d

class LNOConfig:
    MODES = 16
    WIDTH = 8
    NUM_CHANNELS = 3
    SEQUENCE_LENGTH = 10
    INPUT_RES = 6
    OUTPUT_RES = 64
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    MODEL_PATH = 'results_LNO_2dsound/lno_2dsound_reconstruction_6x.pth'

class FNOConfig:
    MODES = 16
    WIDTH = 64
    NUM_CHANNELS = 3
    SEQUENCE_LENGTH = 10
    INPUT_RES = 6
    OUTPUT_RES = 64
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    MODEL_PATH = 'results_FNO_2dsound/fno_2dSound_reconstruction_6x.pth'

def get_grid(shape, device):
    (batchsize, size_x, size_y) = (shape[0], shape[1], shape[2])
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

def load_fno_model(config, path_override=None):
    model = FNO2d(num_channels=config.NUM_CHANNELS, out_channels=config.NUM_CHANNELS * config.SEQUENCE_LENGTH, modes1=config.MODES, modes2=config.MODES, width=config.WIDTH, initial_step=config.SEQUENCE_LENGTH).to(config.DEVICE)
    path = path_override if path_override else config.MODEL_PATH
    if not os.path.exists(path):
        path = os.path.join('results', config.MODEL_PATH)
    if os.path.exists(path):
        state_dict = torch.load(path, map_location=config.DEVICE)
        model.load_state_dict(state_dict)
        print(f'FNO Model loaded from {path}')
    else:
        raise FileNotFoundError(path)
    model.eval()
    return model

def load_lno_model(config, path_override=None):
    tx = torch.linspace(0, 1, config.OUTPUT_RES).unsqueeze(0).to(config.DEVICE)
    ty = torch.linspace(0, 1, config.OUTPUT_RES).unsqueeze(0).to(config.DEVICE)
    model = LNO2d(in_channels=config.NUM_CHANNELS * config.SEQUENCE_LENGTH, out_channels=config.NUM_CHANNELS * config.SEQUENCE_LENGTH, width=config.WIDTH, modes1=config.MODES, modes2=config.MODES, tx=tx, ty=ty).to(config.DEVICE)
    path = path_override if path_override else config.MODEL_PATH
    if not os.path.exists(path):
        path = os.path.join('results', config.MODEL_PATH)
    if os.path.exists(path):
        state_dict = torch.load(path, map_location=config.DEVICE)
        model.load_state_dict(state_dict)
        print(f'LNO Model loaded from {path}')
    else:
        raise FileNotFoundError(path)
    model.eval()
    return model


In [ ]:
data_test_path = 'data/anechoic_high_prec_dataset_last700_test_2.npy'
scalar_path = opt.results_path + '/scalar.npy'
data_raw = np.load(data_test_path)
scalar_data = np.load(scalar_path, allow_pickle=True)
(mean_val, std_val) = (scalar_data[0], scalar_data[1]) if isinstance(scalar_data, np.ndarray) else (scalar_data[0], scalar_data[1])
scaler = DataScalar(mean_val, std_val, device=device)
x_test_raw = select_test_sample_by_freq(data_raw, frequency_hz=375)
x_test = x_test_raw.transpose(0, 1, 4, 2, 3)
print(f'Sample Shape (Model Input): {x_test.shape}')
(_, z) = np.meshgrid(np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), indexing='ij')


In [ ]:
m = 2
T = opt.num_frames
T_prime = 100
B_batch = int(np.ceil((T_prime - m) / (T - m)))
T_prime = B_batch * (T - m) + m
n_samples = 1
seed_samples = 1
rnd = np.random.RandomState(seed_samples)
t_start = 0
samples = x_test[:, t_start:t_start + T_prime]
print(f'Target Shape: {samples.shape}')
samples_scale_all = scaler.normalize(samples)
noise_ratio = 0
noise = rnd.randn(*samples_scale_all.shape)
samples_scale_all = samples_scale_all + noise_ratio * noise
print(f'Applied {noise_ratio * 100}% Gaussian noise to normalized samples.')
print(samples_scale_all.shape)


In [ ]:
outer_loop = 100
denoising_steps1 = outer_loop
denoising_steps2 = 100
seed_sample = 123
cof = 5e-08
obs_type = 'regular'
scale = 6
alpha_case = 0.4
beta = 0.4
gamma = 0.5
sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps1)
sde2 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps2)
net.eval()
reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
setup_seed(seed_sample)
with torch.no_grad():
    if obs_type.lower() == 'random':
        T_prime_y = T_prime
        mask = np.zeros([opt.num_components + opt.num_conditions, opt.image_size, opt.image_size])
        mask[:opt.num_components] = mask_gen([1, opt.image_size, opt.image_size], scale, seed=seed_samples)
        mask_torch = torch.from_numpy(mask).to(device)
        transform_gen = transform = lambda x: x[:, :T_prime_y] * mask[np.newaxis, np.newaxis] if isinstance(x, np.ndarray) else x[:, :T_prime_y] * mask_torch[None, None].to(x.device)
        sparsity = scale
    elif obs_type.lower() == 'regular':
        T_prime_y = T_prime
        transform_gen = transform = lambda x: x[:, :T_prime_y, :opt.num_components, ::scale, ::scale]
        sparsity = 1 - 1 / scale ** 2
    elif obs_type.lower() == 'forecast':
        T_prime_y = 0
        transform_gen = transform = lambda x: x[:, :scale]
        sparsity = 1 - scale / T
    alpha = alpha_case / np.sqrt(1 - sparsity)
    y = transform_gen(samples_scale_all)
    print(f'Observation shape (Normalized): {y.shape}')
    latent_mask = torch.ones([T, 1, 1, 1]).float().to(device)
    obs_mask = torch.zeros([T, 1, 1, 1]).float().to(device)
    frame_indices = torch.arange(T).to(device)
    net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
    print('Sampling SDE...')
    x_inpainted = s3gm_sample_2d(opt, net_fn, sde, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
    x_pred = x_inpainted.copy()
    setup_seed(seed_sample)
    print('Sampling ISS...')
    x_inpainted1 = s3gm_sample_2d_ISS_2dsound(opt, net_fn, sde1, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, reg_coef_pa=10 * T_prime, reg_coef_ar=100.0, denoising_steps=denoising_steps1, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
    x_pred1 = x_inpainted1.copy()
    setup_seed(seed_sample)
    print('Sampling PI-ISS...')
    x_inpainted2 = s3gm_sample_2d_ISS_2dsound_res(opt, net_fn, sde2, y, transform_gen, alpha=1 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, use_residual_objective=False, reg_coef_pa=10 * T_prime, reg_coef_ar=100.0, denoising_steps=denoising_steps2, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True, data_scalar=scaler.unnormalize, physics_start_step=100, reg_coef_wave=1 * cof, reg_coef_cont=1 * cof, reg_coef_euler=1 * cof, physics_c=343.0, physics_rho0=1.21, physics_dt=0.0001, physics_dx=1 / 21, physics_dy=1 / 21)
    x_pred2 = x_inpainted2.copy()
    print('Running LNO inference...')
    try:
        lno_cfg = LNOConfig()
        model_path = 'results_LNO_2dsound/lno_2dsound_reconstruction_6x.pth'
        lno_model = load_lno_model(lno_cfg, path_override=model_path)
        samples_norm_lno = scaler.normalize(samples[:, :, :opt.num_components])
        gt_input_lno = torch.tensor(samples_norm_lno, dtype=torch.float).to(lno_cfg.DEVICE)
        SEQ_LEN = lno_cfg.SEQUENCE_LENGTH
        TARGET_LEN = T_prime
        num_chunks = int(np.ceil(TARGET_LEN / SEQ_LEN))
        pred_lno_list = []
        for i in range(num_chunks):
            chunk_start = i * SEQ_LEN
            chunk_end = chunk_start + SEQ_LEN
            actual_end = min(chunk_end, gt_input_lno.shape[1])
            chunk_data = gt_input_lno[:, chunk_start:actual_end, ...]
            curr_len = chunk_data.shape[1]
            pad_len = 0
            if curr_len < SEQ_LEN:
                pad_len = SEQ_LEN - curr_len
                (B, _, C, H, W) = chunk_data.shape
                padding = torch.zeros(B, pad_len, C, H, W).to(lno_cfg.DEVICE)
                chunk_data_padded = torch.cat([chunk_data, padding], dim=1)
            else:
                chunk_data_padded = chunk_data
            x_low = chunk_data_padded[:, :, :, ::lno_cfg.INPUT_RES, ::lno_cfg.INPUT_RES]
            (B_sz, T_ch, C_ch, H_l, W_l) = x_low.shape
            x_in = x_low.reshape(B_sz * T_ch, C_ch, H_l, W_l)
            x_in = F.interpolate(x_in, size=(lno_cfg.OUTPUT_RES, lno_cfg.OUTPUT_RES), mode='bilinear', align_corners=True)
            x_in_lno = x_in.reshape(B_sz, T_ch, C_ch, lno_cfg.OUTPUT_RES, lno_cfg.OUTPUT_RES)
            x_in_lno = x_in_lno.reshape(B_sz, T_ch * C_ch, lno_cfg.OUTPUT_RES, lno_cfg.OUTPUT_RES)
            out_lno = lno_model(x_in_lno)
            out_lno = out_lno.reshape(B_sz, T_ch, C_ch, lno_cfg.OUTPUT_RES, lno_cfg.OUTPUT_RES)
            if pad_len > 0:
                out_lno = out_lno[:, :-pad_len, ...]
            pred_lno_list.append(out_lno)
        pred_seq_lno = torch.cat(pred_lno_list, dim=1)
        pred_lno_phys = scaler.unnormalize(pred_seq_lno)
        x_pred3_core = pred_lno_phys.detach().cpu().numpy()
        x_pred3 = np.zeros_like(x_pred)
        x_pred3[:, :, :opt.num_components] = x_pred3_core
        if x_pred.shape[2] > opt.num_components:
            x_pred3[:, :, opt.num_components:] = samples[:, :x_pred3.shape[1], opt.num_components:]
    except Exception as e:
        raise
    print('Running FNO inference...')
    try:
        fno_cfg = FNOConfig()
        model_path = 'results_FNO_2dsound/fno_2dSound_reconstruction_6x.pth'
        fno_model = load_fno_model(fno_cfg, path_override=model_path)
        samples_norm = scaler.normalize(samples[:, :, :opt.num_components])
        gt_input = torch.tensor(samples_norm, dtype=torch.float).permute(0, 1, 3, 4, 2).to(fno_cfg.DEVICE)
        SEQ_LEN = fno_cfg.SEQUENCE_LENGTH
        TARGET_LEN = T_prime
        num_chunks = int(np.ceil(TARGET_LEN / SEQ_LEN))
        pred_seq_list = []
        for i in range(num_chunks):
            chunk_start = i * SEQ_LEN
            chunk_end = chunk_start + SEQ_LEN
            actual_end = min(chunk_end, gt_input.shape[1])
            chunk_data = gt_input[:, chunk_start:actual_end, ...]
            curr_len = chunk_data.shape[1]
            pad_len = 0
            if curr_len < SEQ_LEN:
                pad_len = SEQ_LEN - curr_len
                (B, _, H, W, C) = chunk_data.shape
                padding = torch.zeros(B, pad_len, H, W, C).to(fno_cfg.DEVICE)
                chunk_data_padded = torch.cat([chunk_data, padding], dim=1)
            else:
                chunk_data_padded = chunk_data
            x_low = chunk_data_padded[:, :, ::fno_cfg.INPUT_RES, ::fno_cfg.INPUT_RES, :]
            (B_sz, T_ch, H_l, W_l, C_ch) = x_low.shape
            x_in = x_low.reshape(B_sz * T_ch, H_l, W_l, C_ch).permute(0, 3, 1, 2)
            x_in = F.interpolate(x_in, size=(fno_cfg.OUTPUT_RES, fno_cfg.OUTPUT_RES), mode='bilinear', align_corners=True)
            x_in = x_in.reshape(B_sz, T_ch, C_ch, fno_cfg.OUTPUT_RES, fno_cfg.OUTPUT_RES).permute(0, 3, 4, 1, 2).reshape(B_sz, fno_cfg.OUTPUT_RES, fno_cfg.OUTPUT_RES, T_ch * C_ch)
            grid = get_grid(x_in.shape, fno_cfg.DEVICE)
            out = fno_model(x_in, grid).squeeze(-2)
            out = out.view(B_sz, fno_cfg.OUTPUT_RES, fno_cfg.OUTPUT_RES, T_ch, C_ch).permute(0, 3, 1, 2, 4)
            if pad_len > 0:
                out = out[:, :-pad_len, ...]
            pred_seq_list.append(out)
        pred_seq = torch.cat(pred_seq_list, dim=1)
        pred_seq_permuted = pred_seq.permute(0, 1, 4, 2, 3)
        pred_phys = scaler.unnormalize(pred_seq_permuted)
        x_pred4_core = pred_phys.detach().cpu().numpy()
        x_pred4 = np.zeros_like(x_pred)
        x_pred4[:, :, :opt.num_components] = x_pred4_core
        if x_pred.shape[2] > opt.num_components:
            x_pred4[:, :, opt.num_components:] = samples[:, :x_pred4.shape[1], opt.num_components:]
    except Exception as e:
        raise

def unnorm_tensor(x_numpy):
    if x_numpy is None:
        return None
    t = torch.tensor(x_numpy).float().to(device)
    return scaler.unnormalize(t).detach().cpu().numpy()
print('-' * 40)
print('Calculating channel-wise errors...')
phys_pred = unnorm_tensor(x_pred)
phys_pred1 = unnorm_tensor(x_pred1)
phys_pred2 = unnorm_tensor(x_pred2)
phys_pred3 = x_pred3
phys_pred4 = x_pred4
phys_gt = samples[0]

def calc_and_print_fair_metrics(method_name, pred_batch, gt):
    if pred_batch is None:
        return
    pred = pred_batch[0]
    err_p = cal_rmse(gt[:, 0:1], pred[:, 0:1], normalize=True)
    mean_p = np.mean(err_p)
    err_vx = cal_rmse(gt[:, 1:2], pred[:, 1:2], normalize=True)
    mean_vx = np.mean(err_vx)
    err_vy = cal_rmse(gt[:, 2:3], pred[:, 2:3], normalize=True)
    mean_vy = np.mean(err_vy)
    mean_fair = (mean_p + mean_vx + mean_vy) / 3.0
    print(f'{method_name:<20} | {mean_fair:<12.5f} | {mean_p:<10.5f} | {mean_vx:<10.5f} | {mean_vy:<10.5f}')
print('-' * 75)
print(f"{'Method':<20} | {'Mean(Avg)':<12} | {'P(RMSE)':<10} | {'Vx(RMSE)':<10} | {'Vy(RMSE)':<10}")
print('-' * 75)
calc_and_print_fair_metrics('SDE', phys_pred, phys_gt)
calc_and_print_fair_metrics('ISS', phys_pred1, phys_gt)
calc_and_print_fair_metrics('PI-ISS', phys_pred2, phys_gt)
calc_and_print_fair_metrics('LNO', phys_pred3, phys_gt)
calc_and_print_fair_metrics('FNO', phys_pred4, phys_gt)
print('-' * 75)
print('Note: Mean(Avg) = (P + Vx + Vy) / 3. Equal weight for all channels.')
torch.cuda.empty_cache()


In [ ]:
error_curves = {'SDE': {}, 'ISS': {}, 'PI-ISS': {}, 'LNO': {}, 'FNO': {}}

def unnorm_tensor(x_numpy):
    if x_numpy is None:
        return None
    t = torch.tensor(x_numpy).float().to(device)
    return scaler.unnormalize(t).detach().cpu().numpy()
print('-' * 40)
print('Calculating channel-wise errors...')
phys_pred = unnorm_tensor(x_pred)
phys_pred1 = unnorm_tensor(x_pred1)
phys_pred2 = unnorm_tensor(x_pred2)
phys_pred3 = x_pred3
phys_pred4 = x_pred4
phys_gt = samples[0]

def calc_and_print_fair_metrics(method_name, pred_batch, gt, error_dict):
    if pred_batch is None:
        return
    pred = pred_batch[0]
    err_p = cal_rmse(gt[:, 0:1], pred[:, 0:1], normalize=True).squeeze()
    mean_p = np.mean(err_p)
    err_vx = cal_rmse(gt[:, 1:2], pred[:, 1:2], normalize=True).squeeze()
    mean_vx = np.mean(err_vx)
    err_vy = cal_rmse(gt[:, 2:3], pred[:, 2:3], normalize=True).squeeze()
    mean_vy = np.mean(err_vy)
    error_dict[method_name] = {'P': err_p, 'Vx': err_vx, 'Vy': err_vy}
    mean_fair = (mean_p + mean_vx + mean_vy) / 3.0
    print(f'{method_name:<20} | {mean_fair:<12.5f} | {mean_p:<10.5f} | {mean_vx:<10.5f} | {mean_vy:<10.5f}')
print('-' * 75)
print(f"{'Method':<20} | {'Mean(Avg)':<12} | {'P(RMSE)':<10} | {'Vx(RMSE)':<10} | {'Vy(RMSE)':<10}")
print('-' * 75)
calc_and_print_fair_metrics('SDE', phys_pred, phys_gt, error_curves)
calc_and_print_fair_metrics('ISS', phys_pred1, phys_gt, error_curves)
calc_and_print_fair_metrics('PI-ISS', phys_pred2, phys_gt, error_curves)
calc_and_print_fair_metrics('LNO', phys_pred3, phys_gt, error_curves)
calc_and_print_fair_metrics('FNO', phys_pred4, phys_gt, error_curves)
print('-' * 75)
print('Note: Mean(Avg) = (P + Vx + Vy) / 3. Equal weight for all channels.')
torch.cuda.empty_cache()


## Reconstructed pressure and error curves


In [ ]:
(fig, axes) = plt.subplots(1, 6, figsize=(15, 3), constrained_layout=True)
fields = [samples, phys_pred, phys_pred1, phys_pred2, phys_pred3, phys_pred4]
labels = ['Reference', 'SDE', 'ISS', 'PI-ISS', 'LNO', 'FNO']
limit = np.max(np.abs(samples[0, 0, 0]))
for (ax, field, label) in zip(axes, fields, labels):
    im = ax.imshow(field[0, 0, 0], cmap='RdBu_r', vmin=-limit, vmax=limit, origin='lower')
    ax.set_title(label)
fig.colorbar(im, ax=axes, label='Pressure (Pa)', shrink=0.65)
format_current_figure()
fig.savefig(OUTPUT / 'pressure_reconstruction.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
rows = []
for (method, channels) in error_curves.items():
    for (channel, values) in channels.items():
        rows.extend(({'method': method, 'channel': channel, 'frame': i, 'nrmse': float(v)} for (i, v) in enumerate(np.ravel(values))))
pd.DataFrame(rows).to_csv(OUTPUT / 'nrmse_per_frame.csv', index=False)


## Physical residuals


In [ ]:
import numpy as np
import torch
DT = 0.0001
DX = 1 / 21
DY = 1 / 21
RHO_0 = 1.21
C_SOUND = 343.0
C_SQUARED = C_SOUND ** 2
RHO_C_SQUARED = RHO_0 * C_SQUARED

def calculate_physics_residuals(data_field):
    if data_field is None:
        return None
    try:
        p = data_field[0, :, 0, :, :]
        ux = data_field[0, :, 1, :, :]
        uy = data_field[0, :, 2, :, :]
    except IndexError:
        print(f'Cannot extract p, ux and uy from shape {data_field.shape}.')
        return None
    dp_dt = np.gradient(p, DT, axis=0)
    dp_dx = np.gradient(p, DX, axis=2)
    dp_dy = np.gradient(p, DY, axis=1)
    dux_dt = np.gradient(ux, DT, axis=0)
    dux_dx = np.gradient(ux, DX, axis=2)
    duy_dt = np.gradient(uy, DT, axis=0)
    duy_dy = np.gradient(uy, DY, axis=1)
    d2p_dt2 = np.gradient(dp_dt, DT, axis=0)
    d2p_dx2 = np.gradient(dp_dx, DX, axis=2)
    d2p_dy2 = np.gradient(dp_dy, DY, axis=1)
    laplacian_p = d2p_dx2 + d2p_dy2
    res_wave = d2p_dt2 - C_SQUARED * laplacian_p
    divergence_u = dux_dx + duy_dy
    res_cont = dp_dt + RHO_C_SQUARED * divergence_u
    res_mom_x = RHO_0 * dux_dt + dp_dx
    res_mom_y = RHO_0 * duy_dt + dp_dy
    return {'Wave Eq': res_wave, 'Continuity': res_cont, 'Momentum X': res_mom_x, 'Momentum Y': res_mom_y}

def get_residual_stats(residual_dict):
    stats = {}
    if residual_dict is None:
        return {'Error': 'Calculation Failed'}
    for (key, residual_array) in residual_dict.items():
        mae = np.mean(np.abs(residual_array[2:-2, 2:-2, 2:-2]))
        stats[key] = float(mae)
    return stats

def prepare_phys_data(data, needs_unnorm=True):
    if data is None:
        return None
    if needs_unnorm:
        if isinstance(data, np.ndarray):
            t = torch.tensor(data).float().to(device)
        else:
            t = data.float().to(device)
        return scaler.unnormalize(t).detach().cpu().numpy()
    else:
        if isinstance(data, np.ndarray):
            return data
        return data.detach().cpu().numpy()
print('=' * 50)
print('Converting predictions to physical units...')
phys_sde = prepare_phys_data(x_pred, needs_unnorm=True)
phys_iss = prepare_phys_data(x_pred1, needs_unnorm=True)
phys_piiss = prepare_phys_data(x_pred2, needs_unnorm=True)
phys_lno = prepare_phys_data(x_pred3, needs_unnorm=False)
phys_fno = prepare_phys_data(x_pred4, needs_unnorm=False)
phys_ref = prepare_phys_data(samples, needs_unnorm=False)
print('\nCalculating mean absolute PDE residuals...')
methods = [('SDE', phys_sde), ('ISS', phys_iss), ('PI-ISS', phys_piiss), ('LNO', phys_lno), ('FNO', phys_fno), ('Ref.', phys_ref)]
print('-' * 105)
print(f"{'Method':<20} | {'Wave Eq':<18} | {'Continuity':<18} | {'Momentum X':<18} | {'Momentum Y':<18}")
print('-' * 105)
for (name, data) in methods:
    if data is not None:
        residuals = calculate_physics_residuals(data)
        stats = get_residual_stats(residuals)
        w_res = stats.get('Wave Eq', 0)
        c_res = stats.get('Continuity', 0)
        mx_res = stats.get('Momentum X', 0)
        my_res = stats.get('Momentum Y', 0)
        print(f'{name:<20} | {w_res:<18.4e} | {c_res:<18.4e} | {mx_res:<18.4e} | {my_res:<18.4e}')
    else:
        print(f'{name:<20} | Data Not Available')
print('-' * 105)
